In [229]:
import pandas as pd
import glob
import os
import msoffcrypto
import io

In [163]:
all_stock_moved = pd.read_excel("all_stock_moved.xlsx")
# 2026년 자료
all_stocks2026 = all_stock_moved[all_stock_moved['일자'] >= '2026-01-01'].reset_index(drop=True)
all_stocks2026

,No,일자,거래처,납품처,모 델,출하수,단위,다발수,차량No,추가운반비,기준중량(g),출하중량(Kg),단가(원),매출액(원),2.5톤,5.0톤,운반비,운반비합계,출하시간,회차시간
0,10,2026-01-02,동진쎄미켐,동진쎄미켐,동진박스#2(40배),280,2,140,5,NaN,HD1-TOP#2,680.0,520,364000.0,NaN,0.0,NaN,0,2026-01-02 07:25:32.320000,NaN
1,11,2026-01-02,한국이노팩,안성농협식품상품화팀,KURLY-10#2,1050,10,105,588,NaN,HD1-TOP#2,680.0,520,364000.0,0.0,NaN,NaN,0,2026-01-02 07:39:50,NaN
2,12,2026-01-02,CJ대한통운/푸디헤븐,푸디헤븐,KURLY-1#5,1200,20,60,528,NaN,HD1-TOP#2,680.0,520,364000.0,0.0,NaN,NaN,0,2026-01-02 07:47:34.650000,NaN
3,12,2026-01-02,CJ대한통운/푸디헤븐,푸디헤븐,DS1호#2,2200,20,110,NaN,NaN,105,53.0,700,350000.0,NaN,NaN,NaN,NaN,NaN,NaN
4,12,2026-01-02,CJ대한통운/푸디헤븐,푸디헤븐,버섯2K,200,몸통20뚜껑40,NaN,NaN,NaN,HD1-BOT#3,680.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4255,55,2026-04-15,청호나이스,청호나이스,슈퍼6K,258,1,258,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4256,55,2026-04-15,청호나이스,청호나이스,슈퍼V2-BOT,258,1,258,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4257,56,2026-04-15,아풍오닉스/인천,인천,CORNER-PAD,3000,40,75,5,NaN,HD1-TOP#2,680.0,520,364000.0,NaN,0.0,NaN,0,2026-04-15 13:26:19.620000,NaN
4258,57,2026-04-15,SK매직,M100,IAC425 T/B,420,T:8 B:8,NaN,500,NaN,HD1-TOP#2,680.0,520,364000.0,0.0,NaN,NaN,0,2026-04-15 13:35:57.230000,NaN


In [164]:
#생산관리 product들
all_products = pd.read_excel("생산관리_all_product.xlsx")
all_products = all_products[['MODEL(CODE)',
 '제품군',
 "C'TY",
 'C/T',
 '중량',
 '배율',
 'GRADE',
 '포장단위',
 '능율',
 '거래처',
 'T치수LO',
 'T치수UP',
 'B치수LO',
 'B치수UP',
 '단가',
 '규격',
 '두께',
 '끈',
 '색',
 '원MODEL명',
 '최초금형반입일',
 '금형제작처',
 'MODEL(sonata)',
 '금형보관장소',
 '기준배율',
 '체적',
 '금형영구반출일',
 '반출처',
 '비고']]

#팔린거 2026년 출하된거랑 제품 품목 머지해서 만든거 없는건 '단종'으로함 이걸로 product테이블 만들면됨
df_2026_unique_model = pd.DataFrame(all_stocks2026['모  델'].unique())
df_2026_unique_model = df_2026_unique_model.rename(columns={0: "MODEL(CODE)"})
mergemethod = pd.merge(df_2026_unique_model , all_products, on = 'MODEL(CODE)', how='left', indicator= True)
mergemethod.loc[mergemethod['_merge'] == 'left_only', '비고'] = '단종'

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [165]:
mergemethod

,MODEL(CODE),제품군,C'TY,C/T,중량,배율,GRADE,포장단위,능율,거래처,...,최초금형반입일,금형제작처,MODEL(sonata),금형보관장소,기준배율,체적,금형영구반출일,반출처,비고,_merge
0,동진박스#2(40배),BOX,2,120.0,914.0,50,B240NS,2,2.0,동진화성,...,2021-06-03 00:00:00,세일하이테크,동진새미켐4구 소 각인있음,NaN,40,36568.3,NaN,NaN,"석상준과장, 코팅완료",both
1,KURLY-10#2,BOX,3,80.0,280.0,74,SE2500M,10,2.0,삼일이노팩,...,2021-12-29 00:00:00,금형사업부,NaN,NaN,64,17933,NaN,NaN,"석상준과장,코팅완료",both
2,KURLY-1#5,BOX,6,65.0,79.0,74,SE2500M,20,2.0,삼일이노팩,...,2021-02-17 00:00:00,금형사업부,수축주의!!!,NaN,64,4995,NaN,NaN,"석상준과장,코팅완료, 2022.02.07 전체수리",both
3,DS1호#2,BOX,6,65.0,62.0,74,SE2500M,20,2.0,중부스치로폼,...,2021-11-29 00:00:00,금형사업부,"수축,직가래3개",NaN,64,3965,NaN,NaN,"석상준과장,코팅완료",both
4,버섯2K,BOX,3,70.0,104.0,74,B240NS,몸통20뚜껑40,2.0,채인버섯,...,2010-01-13 00:00:00,삼일금형,이형불량 주의!,T-0410,60,6268,NaN,NaN,"2010.01.19승인완료, 2010.03.06기준중량 변경(130g~100g), ...",both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397,언더필840g,SET,3,170.0,340.0,65,SF400,6,2.0,원케미컬,...,2016-07-13 00:00:00,원케미칼이관,수축확인,NaN,NaN,NaN,NaN,NaN,"최재학 사원,2016.07.19사용배율 변경(50->57)",both
398,쥴릭22호(48배),BOX,2,100.0,830.0,58,B240NS,3,2.0,쥴릭,...,2007-11-28 00:00:00,삼일금형,"기타업체 각인 제거, 각인없음 검정끈 묶음!",NaN,NaN,NaN,NaN,NaN,"쥴릭22호 동일금형, 금형없음",both
399,위너-미주향-LOW,SET,2,85.0,327.0,60,B240NS,UP:5 LOW:5,2.0,코웨이,...,2026-04-07 00:00:00,금형제조부,NaN,NaN,60,NaN,NaN,NaN,NaN,both
400,위너-미주향-UP,SET,2,85.0,252.0,60,B240NS,UP:5 LOW:5,2.0,코웨이,...,2026-04-07 00:00:00,금형제조부,NaN,NaN,60,NaN,NaN,NaN,NaN,both


In [279]:
#단가시트

# 납품처랑 단가 null 인애들 삭제
partner_pricelist = pd.read_excel("단가테이블.xlsx", header = 0).dropna(subset=['납품처명', '단가'])
partner_pricelist = partner_pricelist[['납품처명', '삼일모델명', '업체모델명', '단가']]

# 납품처 삼일 단가 까지 똑같은거 없애기
partner_pricelist = partner_pricelist.drop_duplicates(subset=['납품처명', '삼일모델명', '단가'])
partner_pricelist 

,납품처명,삼일모델명,업체모델명,단가
1,IS동서,C554벽걸이양변기,C554벽걸이양변기,9250
2,IS동서,양변기C2000,IW2000,3850
3,NCK,380*320*50T,패드,530
4,NCK,INBOX,드라이 아이스 박스,935
6,NCK,PI(대),PI 수출용 보냉박스(大 ),14307
...,...,...,...,...
5516,인팩코리아/황소걸음,HS-2호(55배),NaN,576
5517,지나농원,NHD-3#3,NaN,1305
5518,지나농원,NHD-4#2,NaN,980
5519,지나농원,HS-2호,NaN,760


In [168]:
naughty_ones = partner_pricelist[partner_pricelist.duplicated(subset=['납품처명', '삼일모델명'], keep= False)].sort_values('납품처명')
naughty_ones

,납품처명,삼일모델명,업체모델명,단가
1932,LG전자,MFZ64518232,NaN,3233
1951,LG전자,MFZ64518232,NaN,3071
4172,다담은,KURLY-1#5,NaN,440
4176,다담은,KURLY-1#5,NaN,460
2607,동광NB,VENETO,NaN,1600
2993,동광NB,VENETO,NaN,1050
4588,로아홈푸드,NHD-3#3,NaN,1070
4592,로아홈푸드,NHD-3#3,NaN,1210
2350,선우,등5호,NaN,1410
2354,선우,등5호,카스텔라,1430


In [ ]:
# duplicated 단가 정재가 필요 
#같은 남품처명, 같은 모델 서로 다른단가
# 어느 단가가 맞는지 알수있는방법은 당월 마감 관리 자료랑 붙혀서 가장 최근에 팔린 모델날짜와 단가를 붙히는방법밖엔없음.


pd.merge(all_stocks2026, naughty_ones, left_on=(['납품처', '모  델']), right_on=(['납품처명', '삼일모델명']))

,No,일자,거래처,납품처,모 델,출하수,단위,다발수,차량No,추가운반비,...,2.5톤,5.0톤,운반비,운반비합계,출하시간,회차시간,납품처명,삼일모델명,업체모델명,단가
0,17,2026-01-27,한화테크윈,한화테크윈,XNP-9300RW,160,T:20B:20,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,한화테크윈,XNP-9300RW,NaN,615
1,17,2026-01-27,한화테크윈,한화테크윈,XNP-9300RW,160,T:20B:20,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,한화테크윈,XNP-9300RW,NaN,595


In [ ]:
# 당월 자료관리 다 가져온거   근데 아까 말햇던것처럼 이게 수원으로가는 final 자료지만 단가가 안맞는것도 있음..
path = r'C:\Users\user\Desktop\회사컴파일들\python_lab\당월자료관리'

all_files = glob.glob(os.path.join(path, '*.xls'))

all_data = []

for file in all_files:
    df = pd.read_excel(file, sheet_name='출하자료')
    df = df[df['일자'].notna()]

    column_i_need = df[['일자', '거래처', '납품처', '모델', '생산수', '출하수','기준중량(g)',
       '출하중량(Kg)', '단가', '매출액']]
    all_data.append(column_i_need)

all_stock = pd.concat(all_data, ignore_index=True)
all_stock

[             일자          거래처    납품처           모델  생산수     출하수  기준중량(g)  \
 0    2025-01-02   중부스티로폼/푸드장    푸드장      JB-N등3호  NaN  1620.0    165.0   
 1    2025-01-02  CJ대한통운/형제식품   형제식품        다용도5K  NaN    80.0    100.0   
 2    2025-01-02  CJ대한통운/형제식품   형제식품  JB4K(몸통/뚜껑)  NaN    30.0    130.0   
 3    2025-01-02  CJ대한통운/형제식품   형제식품   다용도SG-3K-2  NaN    48.0    140.0   
 4    2025-01-02  CJ대한통운/형제식품   형제식품     SI-느타리2K  NaN   240.0     87.0   
 ...         ...          ...    ...          ...  ...     ...      ...   
 2219 2025-01-31          와산업    와산업         CK04  NaN  3000.0     92.0   
 2220 2025-01-31          와산업    와산업     DTS-1000  NaN   150.0    162.0   
 2221 2025-01-31        미래코리아  미래코리아  MFZ67318701  NaN  1000.0    178.0   
 2222 2025-01-31  한국파렛트풀/라라스윗   라라스윗       다용도10K  NaN  1056.0    250.0   
 2223 2025-01-31        한국초저온   송산센터       어상자20K  NaN   600.0    370.0   
 
       출하중량(Kg)      단가        매출액  
 0        267.3   910.0  1474200.0  
 1          8.0   542.0 

In [278]:
partner_pricelist[partner_pricelist['삼일모델명'] == 'XNP-9300RW']

,납품처명,삼일모델명,업체모델명,단가
2864,한화테크윈,XNP-9300RW,NaN,615
2869,한화테크윈,XNP-9300RW,NaN,595


In [220]:
pd.merge(all_stock, naughty_ones, left_on = ['납품처', '모델'], right_on= ['납품처명', '삼일모델명'])

,일자,거래처,납품처,모델,생산수,출하수,기준중량(g),출하중량(Kg),단가_x,매출액,납품처명,삼일모델명,업체모델명,단가_y
0,2025-10-27,한화테크윈,한화테크윈,XNP-9300RW,NaN,100.0,84.0,8.4,1210.0,121000.0,한화테크윈,XNP-9300RW,NaN,615
1,2025-10-27,한화테크윈,한화테크윈,XNP-9300RW,NaN,100.0,84.0,8.4,1210.0,121000.0,한화테크윈,XNP-9300RW,NaN,595
2,2025-12-08,한화테크윈,한화테크윈,XNP-9300RW,NaN,180.0,84.0,15.1,1210.0,217800.0,한화테크윈,XNP-9300RW,NaN,615
3,2025-12-08,한화테크윈,한화테크윈,XNP-9300RW,NaN,180.0,84.0,15.1,1210.0,217800.0,한화테크윈,XNP-9300RW,NaN,595
4,2025-02-05,한화테크윈,한화테크윈,XNP-9300RW,NaN,120.0,84.0,10.1,1190.0,142800.0,한화테크윈,XNP-9300RW,NaN,615
5,2025-02-05,한화테크윈,한화테크윈,XNP-9300RW,NaN,120.0,84.0,10.1,1190.0,142800.0,한화테크윈,XNP-9300RW,NaN,595
6,2025-03-21,한화테크윈,한화테크윈,XNP-9300RW,NaN,120.0,84.0,10.1,1190.0,142800.0,한화테크윈,XNP-9300RW,NaN,615
7,2025-03-21,한화테크윈,한화테크윈,XNP-9300RW,NaN,120.0,84.0,10.1,1190.0,142800.0,한화테크윈,XNP-9300RW,NaN,595
8,2025-04-23,한화테크윈,한화테크윈,XNP-9300RW,NaN,540.0,84.0,45.4,1190.0,642600.0,한화테크윈,XNP-9300RW,NaN,615
9,2025-04-23,한화테크윈,한화테크윈,XNP-9300RW,NaN,540.0,84.0,45.4,1190.0,642600.0,한화테크윈,XNP-9300RW,NaN,595


통합 >>> 단가테이블 >>>> 마감자료

In [276]:
#통합 단가 자료 aka.OLD Price Table

with open("통합170901DATA취합.xlsx", "rb") as f:
    office_file = msoffcrypto.OfficeFile(f)
    office_file.load_key(password="33")
    decrypted = io.BytesIO()
    office_file.decrypt(decrypted)

tonghob = pd.read_excel(decrypted, sheet_name='DATA', header = 1)
tonghob_recol = tonghob.rename(columns={
    'Unnamed: 0': '거래처',
    'Unnamed: 1': '영업담당자',
    'Unnamed: 2': '납품처명',
    'Unnamed: 3':  '삼일모델명',
    'Unnamed: 4': '업체모델명.',
    'Unnamed: 5': 'CODE-No.', 
    'Unnamed: 6': '단가', 
    'Unnamed: 7': '합의중량(g)',
    1: '구단가#1',
    2: '구단가#2',
    3: '구단가#3',
    4: '구단가#4',
    5: '구단가#5',
    6: '구단가#6',
    7: '구단기#7'
})
# 여기서 어느정도 컬럼 정리해놓음 나머지는 미래의 나에게 맞기마..
tonghob_recol  = tonghob_recol[['거래처', '영업담당자', '납품처명', '삼일모델명', '업체모델명.', 'CODE-No.', '단가', '합의중량(g)',
       '구단가#1', '구단가#2', '구단가#3', '구단가#4', '구단가#5', '구단가#6', '구단기#7', '대표자', '사업자번호', '납품주소']]


In [286]:
#일단 지금 필요한거
old_price_table = tonghob_recol[['납품처명',  '삼일모델명', '업체모델명.', '단가','납품주소']].dropna(subset= '납품처명')
old_price_table

,납품처명,삼일모델명,업체모델명.,단가,납품주소
0,(유)돌코리아,NHD-4#2,NaN,780,경기도 평택시 포승읍 평택항만길 305
1,(유)돌코리아,HD4#4,NaN,780,경기도 평택시 포승읍 평택항만길 305
2,(유)돌코리아,JB10K,NaN,1370,경기도 평택시 포승읍 평택항만길 305
3,몬즈컴퍼니,SI-15K,NaN,1640,경기도 안성시 신두만곡로 181
4,(유)돌코리아,SI-5K,NaN,1340,경기도 평택시 포승읍 평택항만길 305
...,...,...,...,...,...
5249,루이스벨라,F3,NaN,3650,충남 천안시 명우리길 9 안쪽 동
5250,루이스벨라,나노박스(64배),NaN,2745,NaN
5251,지나농원,NHD-3#3,NaN,1305,경기 안성시 보개면 진안로 1188번지/동신리 478-4
5252,지나농원,NHD-4#2,NaN,980,NaN


In [293]:
#연간 계휙표_단가
yearly_pricetable = pd.read_excel("연간계휙표_단가.xlsx",header = 1)
#yearly_pricetable[['납품처명',  '삼일모델명', '업체모델명', '단가']]
yearly_pricetable = yearly_pricetable.rename(columns={
    'Unnamed: 0': '거래처',
    'Unnamed: 1': '영업담당자',
    'Unnamed: 2': '납품처명',
    'Unnamed: 3':  '삼일모델명',
    'Unnamed: 4': '업체모델명.',
    'Unnamed: 5': 'CODE-No.', 
    'Unnamed: 6': '단가', 
    'Unnamed: 7': '합의중량(g)',
    1: '구단가#1',
    2: '구단가#2',
    3: '구단가#3',
    4: '구단가#4',
    5: '구단가#5',
    6: '구단가#6',
    7: '구단기#7'
})
yearly_pricetable = yearly_pricetable[['납품처명',  '삼일모델명', '업체모델명.', '단가','납품주소']].dropna(subset= '납품처명')
yearly_pricetable

,납품처명,삼일모델명,업체모델명.,단가,납품주소
1,IS동서,C554벽걸이양변기,C554벽걸이양변기,9250,충남 아산시 탕정면 탕정면로 350-41
2,IS동서,양변기C2000,IW2000,3850,충남 아산시 탕정면 탕정면로 350-41
3,NCK,380*320*50T,패드,530,경기도 평택시 팽성읍 추팔산단로 127
4,NCK,INBOX,드라이 아이스 박스,935,경기도 평택시 팽성읍 추팔산단로 127
5,NCK,PI(대),UN박스(특별주문용) 추가인쇄비 18만원(발주량에 엎어서 계산),NaN,경기도 평택시 팽성읍 추팔산단로 127
...,...,...,...,...,...
5459,인팩코리아/황소걸음,HS-2호(55배),NaN,576,NaN
5460,지나농원,NHD-3#3,NaN,1305,경기 안성시 보개면 진안로 1188번지/동신리 478-4
5461,지나농원,NHD-4#2,NaN,980,NaN
5462,지나농원,HS-2호,NaN,760,NaN


In [282]:
partner_pricelist

,납품처명,삼일모델명,업체모델명,단가
1,IS동서,C554벽걸이양변기,C554벽걸이양변기,9250
2,IS동서,양변기C2000,IW2000,3850
3,NCK,380*320*50T,패드,530
4,NCK,INBOX,드라이 아이스 박스,935
6,NCK,PI(대),PI 수출용 보냉박스(大 ),14307
...,...,...,...,...
5516,인팩코리아/황소걸음,HS-2호(55배),NaN,576
5517,지나농원,NHD-3#3,NaN,1305
5518,지나농원,NHD-4#2,NaN,980
5519,지나농원,HS-2호,NaN,760


그 일단 주소랑 단가 비ㅏ교하기위해서
통합이랑 연간계휙표에있는 단가 비교해볼게요

In [365]:
# 나올 재료: old_price_table (통합), yearly_pricetable (연간)
# 일단 둘이 듀플좀 많을거임
old_price_table  = old_price_table.drop_duplicates()
yearly_pricetable = yearly_pricetable.drop_duplicates()

#스페이스바 없애기
old_price_table['납품처명'] = old_price_table['납품처명'].str.strip()
old_price_table['삼일모델명'] = old_price_table['삼일모델명'].str.strip()

yearly_pricetable['납품처명'] = yearly_pricetable['납품처명'].str.strip()
yearly_pricetable['삼일모델명'] = yearly_pricetable['삼일모델명'].str.strip()


In [371]:
# 납품처 모델
통합_vs_연간 = pd.merge(old_price_table, yearly_pricetable, on = ['납품처명', '삼일모델명'], how = 'outer', indicator= True )
통합_vs_연간

,납품처명,삼일모델명,업체모델명._x,단가_x,납품주소_x,업체모델명._y,단가_y,납품주소_y,_merge
0,(유)돌코리아,HD4#4,NaN,780,경기도 평택시 포승읍 평택항만길 305,NaN,780,경기도 평택시 포승읍 평택항만길 305,both
1,(유)돌코리아,JB10K,NaN,1370,경기도 평택시 포승읍 평택항만길 305,NaN,1370,경기도 평택시 포승읍 평택항만길 305,both
2,(유)돌코리아,NHD-4#2,NaN,780,경기도 평택시 포승읍 평택항만길 305,NaN,780,경기도 평택시 포승읍 평택항만길 305,both
3,(유)돌코리아,SI-15K#4,NaN,2030,경기도 평택시 포승읍 평택항만길 305,NaN,2030,경기도 평택시 포승읍 평택항만길 305,both
4,(유)돌코리아,SI-5K,NaN,1340,경기도 평택시 포승읍 평택항만길 305,NaN,1340,경기도 평택시 포승읍 평택항만길 305,both
...,...,...,...,...,...,...,...,...,...
6178,힘펠,리보-기본,NaN,610,경기도 안성시 신두만곡로 181,NaN,610,경기도 화성시 안녕동 안녕남로 8번길3,both
6179,힘펠,빌트인제습기,NaN,2440,경기도 안성시 신두만곡로 181,NaN,2440,경기도 화성시 안녕동 안녕남로 8번길3,both
6180,힘펠,휴벤S2-1000,NaN,5530,경기도 안성시 신두만곡로 181,NaN,5530,충북 청주시 흥덕구 강내면 다락태성길 25 명성전자,both
6181,힘펠,휴벤S2-400,NaN,3070,경기도 안성시 신두만곡로 181,NaN,3070,충북 청주시 흥덕구 강내면 다락태성길 25 명성전자,both


In [383]:
통합_vs_연간['_merge'] = 통합_vs_연간['_merge'].map({
    'left_only': '통합',
    'right_only': '연간',
    'both': '통합&연간'
})
통합_vs_연간

,납품처명,삼일모델명,업체모델명._x,단가_x,납품주소_x,업체모델명._y,단가_y,납품주소_y,_merge
0,(유)돌코리아,HD4#4,NaN,780,경기도 평택시 포승읍 평택항만길 305,NaN,780,경기도 평택시 포승읍 평택항만길 305,통합&연간
1,(유)돌코리아,JB10K,NaN,1370,경기도 평택시 포승읍 평택항만길 305,NaN,1370,경기도 평택시 포승읍 평택항만길 305,통합&연간
2,(유)돌코리아,NHD-4#2,NaN,780,경기도 평택시 포승읍 평택항만길 305,NaN,780,경기도 평택시 포승읍 평택항만길 305,통합&연간
3,(유)돌코리아,SI-15K#4,NaN,2030,경기도 평택시 포승읍 평택항만길 305,NaN,2030,경기도 평택시 포승읍 평택항만길 305,통합&연간
4,(유)돌코리아,SI-5K,NaN,1340,경기도 평택시 포승읍 평택항만길 305,NaN,1340,경기도 평택시 포승읍 평택항만길 305,통합&연간
...,...,...,...,...,...,...,...,...,...
6178,힘펠,리보-기본,NaN,610,경기도 안성시 신두만곡로 181,NaN,610,경기도 화성시 안녕동 안녕남로 8번길3,통합&연간
6179,힘펠,빌트인제습기,NaN,2440,경기도 안성시 신두만곡로 181,NaN,2440,경기도 화성시 안녕동 안녕남로 8번길3,통합&연간
6180,힘펠,휴벤S2-1000,NaN,5530,경기도 안성시 신두만곡로 181,NaN,5530,충북 청주시 흥덕구 강내면 다락태성길 25 명성전자,통합&연간
6181,힘펠,휴벤S2-400,NaN,3070,경기도 안성시 신두만곡로 181,NaN,3070,충북 청주시 흥덕구 강내면 다락태성길 25 명성전자,통합&연간


In [387]:
통합_vs_연간['납품주소_x'] = 통합_vs_연간['납품주소_y'] 

In [395]:
통합_vs_연간 = 통합_vs_연간.drop(columns = '_merge')
통합_vs_연간

,납품처명,삼일모델명,업체모델명._x,단가_x,납품주소_x,업체모델명._y,단가_y,납품주소_y
0,(유)돌코리아,HD4#4,NaN,780,경기도 평택시 포승읍 평택항만길 305,NaN,780,경기도 평택시 포승읍 평택항만길 305
1,(유)돌코리아,JB10K,NaN,1370,경기도 평택시 포승읍 평택항만길 305,NaN,1370,경기도 평택시 포승읍 평택항만길 305
2,(유)돌코리아,NHD-4#2,NaN,780,경기도 평택시 포승읍 평택항만길 305,NaN,780,경기도 평택시 포승읍 평택항만길 305
3,(유)돌코리아,SI-15K#4,NaN,2030,경기도 평택시 포승읍 평택항만길 305,NaN,2030,경기도 평택시 포승읍 평택항만길 305
4,(유)돌코리아,SI-5K,NaN,1340,경기도 평택시 포승읍 평택항만길 305,NaN,1340,경기도 평택시 포승읍 평택항만길 305
...,...,...,...,...,...,...,...,...
6178,힘펠,리보-기본,NaN,610,경기도 화성시 안녕동 안녕남로 8번길3,NaN,610,경기도 화성시 안녕동 안녕남로 8번길3
6179,힘펠,빌트인제습기,NaN,2440,경기도 화성시 안녕동 안녕남로 8번길3,NaN,2440,경기도 화성시 안녕동 안녕남로 8번길3
6180,힘펠,휴벤S2-1000,NaN,5530,충북 청주시 흥덕구 강내면 다락태성길 25 명성전자,NaN,5530,충북 청주시 흥덕구 강내면 다락태성길 25 명성전자
6181,힘펠,휴벤S2-400,NaN,3070,충북 청주시 흥덕구 강내면 다락태성길 25 명성전자,NaN,3070,충북 청주시 흥덕구 강내면 다락태성길 25 명성전자


In [ ]:
통합_vs_연간 = 통합_vs_연간.drop(columns = '납품주소_y')


In [408]:
통합_vs_연간 = 통합_vs_연간.drop_duplicates()
통합_vs_연간

,납품처명,삼일모델명,업체모델명._x,단가_x,납품주소_x,업체모델명._y,단가_y
0,(유)돌코리아,HD4#4,NaN,780,경기도 평택시 포승읍 평택항만길 305,NaN,780
1,(유)돌코리아,JB10K,NaN,1370,경기도 평택시 포승읍 평택항만길 305,NaN,1370
2,(유)돌코리아,NHD-4#2,NaN,780,경기도 평택시 포승읍 평택항만길 305,NaN,780
3,(유)돌코리아,SI-15K#4,NaN,2030,경기도 평택시 포승읍 평택항만길 305,NaN,2030
4,(유)돌코리아,SI-5K,NaN,1340,경기도 평택시 포승읍 평택항만길 305,NaN,1340
...,...,...,...,...,...,...,...
6178,힘펠,리보-기본,NaN,610,경기도 화성시 안녕동 안녕남로 8번길3,NaN,610
6179,힘펠,빌트인제습기,NaN,2440,경기도 화성시 안녕동 안녕남로 8번길3,NaN,2440
6180,힘펠,휴벤S2-1000,NaN,5530,충북 청주시 흥덕구 강내면 다락태성길 25 명성전자,NaN,5530
6181,힘펠,휴벤S2-400,NaN,3070,충북 청주시 흥덕구 강내면 다락태성길 25 명성전자,NaN,3070


In [421]:
통합_vs_연간['단가_x'] = 통합_vs_연간['단가_x'].fillna(통합_vs_연간['단가_y'])

In [422]:
통합_vs_연간['단가_x'] = pd.to_numeric(통합_vs_연간['단가_x'], errors='coerce')
통합_vs_연간['단가_y'] = pd.to_numeric(통합_vs_연간['단가_y'], errors='coerce')

통합_vs_연간['단가_z'] = 통합_vs_연간['단가_x'] - 통합_vs_연간['단가_y']
통합_vs_연간

,납품처명,삼일모델명,업체모델명._x,단가_x,납품주소_x,업체모델명._y,단가_y,단가_z
0,(유)돌코리아,HD4#4,NaN,780.0,경기도 평택시 포승읍 평택항만길 305,NaN,780.0,0.0
1,(유)돌코리아,JB10K,NaN,1370.0,경기도 평택시 포승읍 평택항만길 305,NaN,1370.0,0.0
2,(유)돌코리아,NHD-4#2,NaN,780.0,경기도 평택시 포승읍 평택항만길 305,NaN,780.0,0.0
3,(유)돌코리아,SI-15K#4,NaN,2030.0,경기도 평택시 포승읍 평택항만길 305,NaN,2030.0,0.0
4,(유)돌코리아,SI-5K,NaN,1340.0,경기도 평택시 포승읍 평택항만길 305,NaN,1340.0,0.0
...,...,...,...,...,...,...,...,...
6178,힘펠,리보-기본,NaN,610.0,경기도 화성시 안녕동 안녕남로 8번길3,NaN,610.0,0.0
6179,힘펠,빌트인제습기,NaN,2440.0,경기도 화성시 안녕동 안녕남로 8번길3,NaN,2440.0,0.0
6180,힘펠,휴벤S2-1000,NaN,5530.0,충북 청주시 흥덕구 강내면 다락태성길 25 명성전자,NaN,5530.0,0.0
6181,힘펠,휴벤S2-400,NaN,3070.0,충북 청주시 흥덕구 강내면 다락태성길 25 명성전자,NaN,3070.0,0.0


In [423]:
통합_vs_연간[통합_vs_연간['단가_z'] != 0] 

,납품처명,삼일모델명,업체모델명._x,단가_x,납품주소_x,업체모델명._y,단가_y,단가_z
9,AJ네트웍스/보라티알,JB10K,NaN,1139.0,경기 이천시 마장면 프리미엄아울렛로 96 보라티알센터,DS1호#2,1290.0,-151.0
10,AJ네트웍스/보라티알,KURLY-1#5,NaN,379.0,경기 이천시 마장면 프리미엄아울렛로 96 보라티알센터,NaN,390.0,-11.0
12,AJ네트웍스/보라티알,SI-4.5K(컬리),NaN,1170.0,NaN,NaN,NaN,NaN
13,AJ네트웍스/보라티알,SI-느타리2K,NaN,461.0,경기 이천시 마장면 프리미엄아울렛로 96 보라티알센터,DW-3호#2,490.0,-29.0
15,AJ네트웍스/보라티알,종합1호,NaN,679.0,경기 이천시 마장면 프리미엄아울렛로 96 보라티알센터,SI-13호,845.0,-166.0
...,...,...,...,...,...,...,...,...
6125,현대포장,SI-느타리2K,NaN,860.0,경가도 평택시 신대동 336-1,NaN,770.0,90.0
6126,현대포장,굴BOX#4,NaN,980.0,NaN,NaN,NaN,NaN
6130,현진로지스,SI-15K#4,구)삼경프라자,2770.0,경기도 이천시 대월면 대장로 187 3층,구)삼경프라자,3180.0,-410.0
6155,훈이네마늘빵,SI-N4K,NaN,1070.0,NaN,NaN,NaN,NaN


In [339]:
old_price_table

,납품처명,삼일모델명,업체모델명.,단가,납품주소
0,(유)돌코리아,NHD-4#2,NaN,780,경기도 평택시 포승읍 평택항만길 305
1,(유)돌코리아,HD4#4,NaN,780,경기도 평택시 포승읍 평택항만길 305
2,(유)돌코리아,JB10K,NaN,1370,경기도 평택시 포승읍 평택항만길 305
3,몬즈컴퍼니,SI-15K,NaN,1640,경기도 안성시 신두만곡로 181
4,(유)돌코리아,SI-5K,NaN,1340,경기도 평택시 포승읍 평택항만길 305
...,...,...,...,...,...
5249,루이스벨라,F3,NaN,3650,충남 천안시 명우리길 9 안쪽 동
5250,루이스벨라,나노박스(64배),NaN,2745,NaN
5251,지나농원,NHD-3#3,NaN,1305,경기 안성시 보개면 진안로 1188번지/동신리 478-4
5252,지나농원,NHD-4#2,NaN,980,NaN


In [347]:
old_price_table[old_price_table['납품처명'].str.contains('그린라인')].iloc[0]['납품처명']


' 그린라인유한회사'

In [333]:
통합_vs_연간[통합_vs_연간['_merge'] == 'left_only'].sample(10)

,납품처명,삼일모델명,업체모델명._x,단가_x,납품주소_x,업체모델명._y,단가_y,납품주소_y,_merge
4121,인팩코리아,DW-3호#2,NaN,1512,NaN,NaN,NaN,NaN,left_only
4165,인팩코리아,NHD-3#3,NaN,913,NaN,NaN,NaN,NaN,left_only
4126,인팩코리아,F5,NaN,1030,NaN,NaN,NaN,NaN,left_only
2085,본수원갈비(본점),SI다용도,NaN,2310,NaN,NaN,NaN,NaN,left_only
2677,싱싱패키지/건어맨,SI-4호,NaN,556,NaN,NaN,NaN,NaN,left_only
2816,싱싱패키지/진원,나노박스(64배),NaN,2475,NaN,NaN,NaN,NaN,left_only
26,AJ네트웍스/정들,HD1-TOP#2,NaN,421,NaN,NaN,NaN,NaN,left_only
2336,삼일이노팩&탭스인터내셔널,TNPP(IN-EPS),NaN,NaN,NaN,NaN,NaN,NaN,left_only
4163,인팩코리아,KURLY-3#8,NaN,907,NaN,NaN,NaN,NaN,left_only
4024,이지패키지/요즘,굴BOX#4,NaN,588,NaN,NaN,NaN,NaN,left_only
